In [ ]:
# for manipulating the PDF
import fitz

# for OCR using PyTesseract
import cv2                              # pre-processing images
import pytesseract                      # extracting text from images
import numpy as np
import matplotlib.pyplot as plt         # displaying output images

from PIL import Image

In [ ]:
# global variables (replace with you own file)
SCANNED_FILE = "../data/in/Scanned - Multi-Column Single Page Document.pdf"

img = cv2.imread(SCANNED_FILE)

In [ ]:
zoom_x = 2.0 # horizontal zoom
zoom_y = 2.0 # vertical zoom
mat = fitz.Matrix(zoom_x, zoom_y)

In [ ]:
doc = fitz.open(SCANNED_FILE)

print("Generated pages: ")
for page in doc:
    pix = page.get_pixmap(matrix=mat)
    png = '..\\data\\out\\input-' + SCANNED_FILE.split('\\')[-1].split('.')[0] + 'page-%i.png' % page.number
    print(png)
    pix.save(png)

original_image = cv2.imread('..\data\out\input-page-0.png')

In [ ]:
# convert the image to grayscale
gray_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2GRAY)
plt.figure(figsize=(25, 15))
plt.imshow(gray_image, cmap='gray')
plt.show()

In [ ]:
# Performing OTSU threshold
ret, threshold_image = cv2.threshold(gray_image, 0, 255, cv2.THRESH_OTSU | cv2.THRESH_BINARY_INV)
plt.figure(figsize=(25, 15))
plt.imshow(threshold_image, cmap='gray')
plt.show()

In [ ]:
rectangular_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (66, 66))

# Applying dilation on the threshold image
dilated_image = cv2.dilate(threshold_image, rectangular_kernel, iterations = 1)
plt.figure(figsize=(25, 15))
plt.imshow(dilated_image)
plt.show()

# Finding contours
contours, hierarchy = cv2.findContours(dilated_image, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

# Creating a copy of the image
copied_image = original_image.copy()

with open("../data/out/recognized-kernel-66-66.txt", "w+") as f:
    f.write("")
f.close()

mask = np.zeros(original_image.shape, np.uint8)
 
# Looping through the identified contours
# Then rectangular part is cropped and passed on to pytesseract
# pytesseract extracts the text inside each contours
# Extracted text is then written into a text file
for cnt in contours:
    x, y, w, h = cv2.boundingRect(cnt)
     
    # Cropping the text block for giving input to OCR
    cropped = copied_image[y:y + h, x:x + w]
    
    with open("../data/out/recognized-kernel-66-66.txt", "a") as f:
        # Apply OCR on the cropped image
        text = pytesseract.image_to_string(cropped, lang='lat', config='--oem 3 --psm 1')
        print(text)
        
    masked = cv2.drawContours(mask, [cnt], 0, (255, 255, 255), -1)

plt.figure(figsize=(25, 15))
plt.imshow(masked, cmap='gray')
plt.show()